In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
from email.mime import text
import json
from pyexpat.errors import messages


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [4]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that parses an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path' and returns just the bucket name."}, {'task': "Create a JSON object that represents an AWS IAM policy statement allowing read-only access to a specific S3 bucket named 'my-data-bucket'."}, {'task': "Write a regular expression that matches valid AWS EC2 instance IDs, which follow the pattern 'i-' followed by 17 hexadecimal characters (e.g., i-0a1b2c3d4e5f6g7h8)."}]


In [5]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    """
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [8]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [ ]:
import json
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses an AWS S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse an AWS S3 bucket name from an S3 URI.\n    \n    Args:\n        s3_uri: S3 URI in the format 's3://bucket-name/key/path'\n        \n    Returns:\n        The bucket name extracted from the S3 URI\n        \n    Raises:\n        ValueError: If the S3 URI format is invalid\n    \"\"\"\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}. Must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    remaining = s3_uri[5:]\n    \n    # Extract bucket name (everything before the first '/')\n    bucket_name = remaining.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}. No bucket name found.\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"__main__\":\n   